In [2]:
!pip install emoji
!pip install emoji transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 9.2 MB/s eta 0:00:00


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("pypiahmad/goodreads-book-reviews1")

100%|██████████| 8.14G/8.14G [01:56<00:00, 75.0MB/s]

Extracting files...


In [5]:
file_path = path + "/goodreads_reviews_dedup.json"

In [7]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [8]:
# ================================
# 1. IMPORTS
# ================================
import pandas as pd
import numpy as np
import re

import emoji

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

from transformers import pipeline

In [31]:


# ================================
# 2. DOWNLOAD NLTK DATA
# ================================
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# ================================
# 3. LOAD DATASET
# ================================
df = pd.read_json(file_path, lines=True, nrows=50000)

df = df[['review_text', 'rating']].dropna()

# ================================
# 4. LABEL CREATION
# ================================
def convert_sentiment(rating):
    if rating >= 4:
        return "positive"
    elif rating == 3:
        return "neutral"
    else:
        return "negative"

df['sentiment'] = df['rating'].apply(convert_sentiment)

# ================================
# 5. PREPROCESSING
# ================================
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()

    # keep basic punctuation (important for sentiment)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    tokens = word_tokenize(text)

    tokens = [w for w in tokens if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]

    return " ".join(tokens)

df['clean_review'] = df['review_text'].apply(preprocess_text)

# ================================
# 6. TF-IDF + SVM MODEL
# ================================
vectorizer = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1,2),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True
)

neutral_words = ["okay", "fine", "average", "normal", "not bad", "not great"]

def boost_neutral(text):
    for word in neutral_words:
        if word in text:
            return text + " neutral"
    return text

df['clean_review'] = df['review_text'].apply(lambda x: preprocess_text(boost_neutral(x)))

X = vectorizer.fit_transform(df['clean_review'])
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



# ================================
# 9. EMOJI DETECTION FUNCTION
# ================================


# ================================
# 10. FINAL HYBRID PREDICTION
# ================================


# ================================
# 11. TEST CASES
# ================================


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [44]:
model = LinearSVC(class_weight='balanced')
model.fit(X_train, y_train)

# ================================
# 7. EVALUATION
# ================================
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# ================================
# 8. TRANSFORMER MODEL (fallback)
# ================================
transformer_model = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Accuracy: 0.682
              precision    recall  f1-score   support

    negative       0.49      0.51      0.50      1415
     neutral       0.44      0.42      0.43      2204
    positive       0.81      0.81      0.81      6381

    accuracy                           0.68     10000
   macro avg       0.58      0.58      0.58     10000
weighted avg       0.68      0.68      0.68     10000



Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [45]:
import joblib

# Save model + vectorizer
joblib.dump({
    "model": model,
    "vectorizer": vectorizer
}, "sentiment_model.pkl")

print("Model saved")

Model saved


In [46]:


data = joblib.load("sentiment_model.pkl")

model = data["model"]
vectorizer = data["vectorizer"]

print("Model loaded")

Model loaded


In [47]:
def contains_emoji(text):
    return any(char in emoji.EMOJI_DATA for char in text)

In [48]:
def predict_sentiment(text):

    if not text.strip():
        return "neutral"

    # If emoji present → use transformer
    if contains_emoji(text):
        result = transformer_model(text)[0]

        label = result['label'].lower()

        # Normalize labels
        if "pos" in label:
            return "positive"
        elif "neg" in label:
            return "negative"
        else:
            return "neutral"

    # Else use SVM
    cleaned = preprocess_text(text)
    vec = vectorizer.transform([cleaned])

    if vec.nnz == 0:
        return "neutral"

    return model.predict(vec)[0]

['sentiment_pipeline.pkl']

In [ ]:
print(predict_sentiment("This product is amazing"))
print(predict_sentiment("I love this 😍"))
print(predict_sentiment("😡"))
print(predict_sentiment("😟"))
print(predict_sentiment("The book is okay"))


positive
positive
negative
negative
neutral
